In [ ]:
!pip install torch transformers peft bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Model dosyasını çıkarıyoruz (zaten models/sql-mistral-lora şeklinde çıkacaktır)
!unzip -o -q lora_model.zip

# Veritabanını çıkartıp doğru klasöre taşıyoruz
!unzip -o -q spider_data.zip
!mkdir -p data/database
!cp -r spider_data/database/* data/database/
# (Eğer Kaggle'dan farklı bir zip indirdiyseniz cp komutundaki 'spider/database/*' kısmı değişebilir)

In [ ]:
# Modeli yükle
!unzip -o -q /content/drive/MyDrive/lora_model.zip -d .

# Veritabanını çıkartıp doğru klasöre taşıyoruz
!unzip -o -q /content/drive/MyDrive/spider_data.zip -d .
!mkdir -p data/database
!cp -r spider_data/database/* data/database/

In [ ]:
import os
import sqlite3
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --- YOLLAR ---
BASE_MODEL = "mistralai/Mistral-7B-v0.1"
LORA_MODEL_DIR = "models/sql-mistral-lora"
DB_DIR = "data/database"

def execute_sql(db_id, sql_query):
    """Sorguyu veritabanında çalıştırır."""
    db_path = os.path.join(DB_DIR, db_id, f"{db_id}.sqlite")
    if not os.path.exists(db_path):
        return f"DB_NOT_FOUND: {db_path}"
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql_query)
        result = cursor.fetchall()
        conn.close()
        return set(result)
    except Exception as e:
        return f"SQL_HATA: {e}"

def main():
    print("=" * 70)
    print("🚀 Model Colab GPU'sunda Test Ediliyor...")
    print("=" * 70)

    # 4-Bit Sıkıştırma
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    print("[1/3] Tokenizer yükleniyor...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    print("[2/3] Mistral-7B 4-bit yükleniyor...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto"
    )

    print("[3/3] LoRA adaptörleri modele giydiriliyor...")
    model = PeftModel.from_pretrained(base_model, LORA_MODEL_DIR)

    print("\n✅ Sistem Hazır! Soru soruluyor...\n")

    # --- TEST SENARYOSU ---
    test_db = "department_management"
    schema = "CREATE TABLE department (Department_ID number, Name text, Creation text, Ranking number, Budget_in_Billions number, Num_Employees number);"
    question = "How many departments are there?"
    hedef_sql = "SELECT count(*) FROM department"

    prompt = f"Sen bir SQL uzmanısın. Aşağıdaki tablo şemasına bakarak sorulan soruyu cevaplayan SQL sorgusunu yaz.\n\nŞema: {schema}\nSoru: {question}\nCevap:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    print(f"Soru: {question}")
    print("🧠 Model düşünüyor...")

    outputs = model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id)
    uretilen_metin = tokenizer.decode(outputs[0], skip_special_tokens=True)
    uretilen_sql = uretilen_metin.split("Cevap:")[-1].split(";")[0].strip()

    print("-" * 60)
    print(f"🎯 Beklenen Hedef SQL : {hedef_sql}")
    print(f"🤖 Modelin Ürettiği SQL: {uretilen_sql}")
    print("-" * 60)

    hedef_sonuc = execute_sql(test_db, hedef_sql)
    model_sonuc = execute_sql(test_db, uretilen_sql)

    print(f"Hedef Çıktı : {hedef_sonuc}")
    print(f"Model Çıktı : {model_sonuc}")

    if hedef_sonuc == model_sonuc and "HATA" not in str(model_sonuc):
        print("\n🏆 SONUÇ: BAŞARILI! (Execution Accuracy: 1)")
    else:
        print("\n❌ SONUÇ: BAŞARISIZ! Veriler uyuşmuyor.")

main()

🚀 Model Colab GPU'sunda Test Ediliyor...
[1/3] Tokenizer yükleniyor...
[2/3] Mistral-7B 4-bit yükleniyor...


KeyboardInterrupt: 

In [ ]:
import os
import sqlite3
import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --- AYARLAR VE YOLLAR ---
BASE_MODEL = "mistralai/Mistral-7B-v0.1"
LORA_MODEL_DIR = "models/sql-mistral-lora"
DB_DIR = "data/database"
TEST_SAYISI = 100

print("=" * 70)
print("🚀 SİSTEM HAZIRLIK AŞAMASI")
print("=" * 70)

# 1. MODEL VE TOKENIZER YÜKLEME (Eksik olan kısım buydu!)
print("[1/3] Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("[2/3] Mistral-7B temel modeli 4-bit olarak GPU'ya yükleniyor (Bu biraz sürebilir)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

print("[3/3] Eğittiğin LoRA adaptörleri modele giydiriliyor...")
model = PeftModel.from_pretrained(base_model, LORA_MODEL_DIR)
print("✅ Yapay Zeka Beyni GPU'ya Yerleşti!\n")

# --- FONKSİYONLAR ---
def get_db_schema(db_id):
    """Veritabanına bağlanıp CREATE TABLE şemalarını otomatik çeker."""
    db_path = os.path.join(DB_DIR, db_id, f"{db_id}.sqlite")
    if not os.path.exists(db_path):
        return ""
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        conn.close()
        return "\n".join([t[0] for t in tables if t[0] is not None])
    except:
        return ""

def execute_sql(db_id, sql_query):
    """Sorguyu çalıştırıp sonucu döndürür."""
    db_path = os.path.join(DB_DIR, db_id, f"{db_id}.sqlite")
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql_query)
        result = cursor.fetchall()
        conn.close()
        return set(result)
    except Exception as e:
        return f"HATA"

# --- TEST DÖNGÜSÜ ---
print("=" * 70)
print(f"📊 {TEST_SAYISI} SORULUK TOPLU TEST BAŞLIYOR...")
print("=" * 70)

val_dataset = load_dataset("xlangai/spider", split="validation")
dogru_sayisi = 0
hata_sayisi = 0

for i in range(TEST_SAYISI):
    ornek = val_dataset[i]
    db_id = ornek["db_id"]
    soru = ornek["question"]
    hedef_sql = ornek["query"]

    schema = get_db_schema(db_id)
    if not schema:
        continue

    prompt = f"Sen bir SQL uzmanısın. Aşağıdaki tablo şemasına bakarak sorulan soruyu cevaplayan SQL sorgusunu yaz.\n\nŞema:\n{schema}\n\nSoru: {soru}\nCevap:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # max_new_tokens değerini biraz artırdık ki uzun SQL'ler yarım kesilmesin
    outputs = model.generate(**inputs, max_new_tokens=80, pad_token_id=tokenizer.eos_token_id)
    uretilen_metin = tokenizer.decode(outputs[0], skip_special_tokens=True)
    uretilen_sql = uretilen_metin.split("Cevap:")[-1].split(";")[0].strip()

    hedef_sonuc = execute_sql(db_id, hedef_sql)
    model_sonuc = execute_sql(db_id, uretilen_sql)

    if hedef_sonuc == model_sonuc and model_sonuc != "HATA":
        dogru_sayisi += 1
        durum = "✅ DOĞRU"
    else:
        hata_sayisi += 1
        durum = "❌ YANLIŞ"

    print(f"[{i+1}/{TEST_SAYISI}] DB: {db_id} | {durum}")

# --- FİNAL RAPORU ---
basari_orani = (dogru_sayisi / TEST_SAYISI) * 100

print("\n" + "=" * 70)
print("🏆 TEST TAMAMLANDI! FİNAL RAPORU 🏆")
print("=" * 70)
print(f"Toplam Test Edilen Soru : {TEST_SAYISI}")
print(f"Doğru Çalışan Sorgular  : {dogru_sayisi}")
print(f"Yanlış veya Hatalı      : {hata_sayisi}")
print(f"EXECUTION ACCURACY      : %{basari_orani:.2f}")
print("=" * 70)

🚀 SİSTEM HAZIRLIK AŞAMASI
[1/3] Tokenizer yükleniyor...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[2/3] Mistral-7B temel modeli 4-bit olarak GPU'ya yükleniyor (Bu biraz sürebilir)...


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]